# 1.8 猜錯要付出多少代價？


兩次預測都把「看」排第一，第一回給它 0.5，第二回給它 0.9。如果答案確實是「看」，第二回應算更好，因為它把更多機會放到正確答案上。只記「有沒有選對第一名」看不出這個差別；訓練需要一個能隨信心連續改變的代價，才知道分數調整是進步還是退步。

接續 [1.7](https://birdhackor.github.io/tiny-perceptron-vlm/1.7.html) 的機率分配，我們只取正確答案那一格的機率 p，再計算 `-log(p)`。代價也叫 loss；這個名稱是在說「希望降低的數字」，並不是猜錯題數。[W.4](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.4) 解釋了對數與數值直覺：給正確答案 0.5，代價約 0.693；給 0.9，代價約 0.105；給 0.1，代價約 2.303。因此越有把握地忽略正確答案，處罰越大。

如果一批例子每個都有唯一正確類別，把它們的負對數代價取平均，通常叫交叉熵（cross entropy）。這裡不需要先學整套資訊理論，只要能追到「正確答案的機率是哪一格」。自然對數的代價單位叫 nat；除以 `log(2)` 才換成 bit。換單位不改誰比較好。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch.nn import functional as F

p = torch.tensor([0.1, 0.5, 0.9])
print("正確機率", p, "代價", -p.log())
logits = torch.tensor([[0.0, 2.0, 0.0]])
target = torch.tensor([1])
probabilities = logits.softmax(dim=-1)
manual = -probabilities[0, 1].log()
loss = F.cross_entropy(logits, target)
print(probabilities, manual.item(), loss.item())
assert torch.allclose(loss, manual)

第一段輸入三個正確機率，直接列出上述代價。第二段是一個問題、三個候選分數，形狀 `[1,3]`；這種未換成機率的分數常叫 logits。`target=[1]` 表示這一題的答案是索引 1 的候選，也就是從左數第二個，索引仍從 0 開始。`probabilities[0,1]` 因而取第一題的正確機率。分配約 `[0.1065,0.7870,0.1065]`，手算負對數和內建交叉熵都約 0.2395。

`F` 是給 `torch.nn.functional` 取的簡短名字，裡面提供直接計算的函式。`F.cross_entropy` 接收原始 logits 和答案 ID，內部已用數值穩定的方式處理 softmax 與對數；不應把 `probabilities` 再送進它。否則機率會被當成另一組分數再換算一次，得到不同代價。這個介面也不用輸入完整的正確機率表，只需一個答案位置。

比較平均 loss 要說清楚在哪份資料、哪種文字拆分上。相同文字若被拆成不同數量的片段，每片段平均的意義就不同；第 6 章會討論更公平的比較。現在先固定字表與例子，讓代價變化可以歸因於分數變化。

練習只把 `target` 改成 `[0]`，並把手算索引 `[0,1]` 改成 `[0,0]`。先預測代價會高於還是低於 0.2395，再核對應約 2.2395。分數表完全沒變，只換了哪一格是答案；這說明 loss 同時需要預測與真實答案，光看分數高低無法判斷好壞。
